# Using LLMs and JSON APIs in Humanities Research

This notebook is for first-year Digital Humanities master's students with minimal programming experience. It introduces JSON, web APIs, and practical LLM use through OpenRouter.

The notebook is designed for Google Colab first, but it also works locally in Jupyter. The JSON examples run without an API key. The OpenRouter examples are skipped gracefully until you provide an API key.

![JSON API and LLM workflow](https://raw.githubusercontent.com/ValRCS/RTU_Python_DE0800/master/img/JSON_API_LLM_WORKFLOW.png)

## Learning Goals

By the end of this lesson you should be able to:

- recognize JSON objects, arrays, strings, numbers, booleans, and null values
- convert between Python data structures and JSON text
- understand the parts of an API request: URL, headers, authentication, and JSON body
- send a small request to an LLM API through OpenRouter
- ask an LLM to return structured JSON for a humanities research task
- parse, inspect, and save the results

Core sections are intended for a 2 x 45 minute lesson. Sections marked **Bonus** are for continued exploration.

## Colab and Local Setup

In Colab, most required libraries are already installed. Locally, make sure your Python environment has `requests` available.

This setup cell imports common libraries and prints enough environment information to make debugging easier.

In [1]:
from pathlib import Path
from datetime import datetime
import json # important here since we will be serializing and de-serializing JSON data from text into Python data structures and back
import os
import sys

try:
    import requests # this is external library for web requests / scraping
except ImportError:
    print("The requests library is missing. Install it with: python -m pip install requests")
    raise

try:
    import pandas as pd
except ImportError:
    pd = None

print(f"Python version: {sys.version.split()[0]}")
print(f"Current time: {datetime.now().isoformat(timespec='seconds')}")
print(f"Current working directory: {Path.cwd()}")
print(f"requests version: {requests.__version__}")
print(f"pandas available: {pd is not None}")
# if pandas exists print pandas version
if pd:
    print(f"pandas version: {pd.__version__}")

Python version: 3.12.13
Current time: 2026-05-20T13:50:18
Current working directory: /content
requests version: 2.32.4
pandas available: True
pandas version: 2.2.2


## API Key Setup

OpenRouter uses an API key for authentication. Do not paste your key into a notebook cell as plain text.

This cell looks for the key in three places:

1. Google Colab secrets, under the name `OPENROUTER_API_KEY`
2. a local environment variable named `OPENROUTER_API_KEY`
3. a hidden prompt using `getpass`

If you press Enter at the prompt, the API examples will be skipped and the JSON examples will still work.

In [2]:
def running_in_colab():
    try:
        import google.colab  # type: ignore
        # importing google.colab means we are in colab, no one else uses this library
        return True
    except ImportError:
        return False

IN_COLAB = running_in_colab()
OPENROUTER_API_KEY = None

if IN_COLAB:
    print("We are running in Colab will try to check secret from there")
    try:
        from google.colab import userdata  # type: ignore
        OPENROUTER_API_KEY = userdata.get("OPEN_ROUTER_API_KEY")
    except Exception:
        OPENROUTER_API_KEY = None

# plan B get key from system (Operating Sytem) environment
if not OPENROUTER_API_KEY:
    OPENROUTER_API_KEY = os.environ.get("OPEN_ROUTER_API_KEY")

if not OPENROUTER_API_KEY:
    import getpass
    entered_key = getpass.getpass("Enter OpenRouter API key, or press Enter to skip API calls: ").strip()
    OPENROUTER_API_KEY = entered_key or None

HAS_OPENROUTER_KEY = bool(OPENROUTER_API_KEY)
print("OpenRouter API key available:", HAS_OPENROUTER_KEY)

We are running in Colab will try to check secret from there
OpenRouter API key available: True


## What Is JSON?

JSON means JavaScript Object Notation. It is a text format for structured data.

Even though JSON comes from JavaScript, Python can work with it very easily. In APIs, JSON is often used for both the request we send and the response we receive.

A JSON object looks similar to a Python dictionary:

```json
{
  "title": "Digital Humanities Methods",
  "year": 2026,
  "topics": ["text analysis", "metadata", "visualization"]
}
```

In [3]:
book_metadata = {
    "title": "Digital Humanities Methods",
    "year": 2026,
    "topics": ["text analysis", "metadata", "visualization"],
    "published": True,
    "notes": None,
}

print(type(book_metadata))
print(book_metadata)

<class 'dict'>
{'title': 'Digital Humanities Methods', 'year': 2026, 'topics': ['text analysis', 'metadata', 'visualization'], 'published': True, 'notes': None}


## Python Data to JSON Text

The Python `json` module converts between Python values and JSON text.

Useful functions:

- `json.dumps(data)` converts Python data to a JSON string
- `json.loads(text)` converts a JSON string to Python data
- `json.dump(data, file)` writes JSON to a file
- `json.load(file)` reads JSON from a file

In [5]:
# we will create easily humanly readable text from our PYthon data structure (in this case dictionary)
json_text = json.dumps(book_metadata, indent=2, ensure_ascii=False)
# ensure_ascii is so we see non English characters in text instead of something like \u0x05 which is hard to read for humans :0
print(json_text)
print(type(json_text))

{
  "title": "Digital Humanities Methods",
  "year": 2026,
  "topics": [
    "text analysis",
    "metadata",
    "visualization"
  ],
  "published": true,
  "notes": null
}
<class 'str'>


In [6]:
# if we have text in JSON format we can get Python data structure back
parsed_book = json.loads(json_text)
print(parsed_book)
print(type(parsed_book))
print("Title:", parsed_book["title"])
print("First topic:", parsed_book["topics"][0])

{'title': 'Digital Humanities Methods', 'year': 2026, 'topics': ['text analysis', 'metadata', 'visualization'], 'published': True, 'notes': None}
<class 'dict'>
Title: Digital Humanities Methods
First topic: text analysis


## Common JSON Mistakes

Strict JSON is less flexible than Python.

Important differences:

- JSON strings must use double quotes
- JSON has `true` and `false`, not Python `True` and `False`
- JSON has `null`, not Python `None`
- JSON does not allow trailing commas
- JSON does not allow comments

In [7]:
bad_examples = [
    "{'title': 'Single quotes are not valid JSON'}",
    '{"title": "Trailing comma",}',
    '{"published": True}',
    '{"notes": None}',
]

for bad_json in bad_examples:
    print("Trying:", bad_json)
    try:
        json.loads(bad_json)
        print("Parsed successfully")
    except json.JSONDecodeError as error:
        print("JSON error:", error)
    print()

Trying: {'title': 'Single quotes are not valid JSON'}
JSON error: Expecting property name enclosed in double quotes: line 1 column 2 (char 1)

Trying: {"title": "Trailing comma",}
JSON error: Expecting property name enclosed in double quotes: line 1 column 28 (char 27)

Trying: {"published": True}
JSON error: Expecting value: line 1 column 15 (char 14)

Trying: {"notes": None}
JSON error: Expecting value: line 1 column 11 (char 10)



## JSON for Humanities Data

JSON works well for humanities metadata because it can represent nested information: a document can have a title, date, language, source, people, places, and analysis results.

In [8]:
document_record = {
    "document_id": "sample-letter-001",
    "title": "Letter from a Student in Riga",
    "date": "1924-03-18",
    "language": "English",
    "source": {
        "archive": "Course sample collection",
        "format": "transcribed text",
    },
    "analysis": {
        "people": ["Anna", "Marta"],
        "places": ["Riga", "Jelgava"],
        "themes": ["education", "travel", "family"],
    },
}

print(json.dumps(document_record, indent=2, ensure_ascii=False))

{
  "document_id": "sample-letter-001",
  "title": "Letter from a Student in Riga",
  "date": "1924-03-18",
  "language": "English",
  "source": {
    "archive": "Course sample collection",
    "format": "transcribed text"
  },
  "analysis": {
    "people": [
      "Anna",
      "Marta"
    ],
    "places": [
      "Riga",
      "Jelgava"
    ],
    "themes": [
      "education",
      "travel",
      "family"
    ]
  }
}


## Saving and Loading JSON Files

APIs are not useful only while the notebook is running. We often want to save results and use them later.

In [9]:
output_path = Path("sample_document_record.json")

with output_path.open("w", encoding="utf-8") as file:
    json.dump(document_record, file, indent=2, ensure_ascii=False)

with output_path.open("r", encoding="utf-8") as file:
    loaded_record = json.load(file)

print("Saved to:", output_path.resolve())
print("Loaded title:", loaded_record["title"])

Saved to: /content/sample_document_record.json
Loaded title: Letter from a Student in Riga


In [10]:
# let me download this json file to my local computer
from google.colab import files
files.download(output_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## What Is an API?

An API is a structured way for one program to ask another program for something.

For a web API request we usually need:

- a URL, such as `https://openrouter.ai/api/v1/chat/completions`
- headers, which contain metadata such as authentication and content type
- a JSON body, which contains the actual request data
- a response, usually also JSON

OpenRouter provides access to many LLM providers through one API. Its chat completion API follows an OpenAI-style message structure.

Reference links:

- OpenRouter API overview: https://openrouter.ai/docs/api/reference/overview/
- Chat completion endpoint: https://openrouter.ai/docs/api/api-reference/chat/send-chat-completion-request
- Models endpoint: https://openrouter.ai/docs/api/api-reference/models/get-models

## Anatomy of an LLM API Request

Before we send anything to the internet, inspect the JSON payload locally.

The important part is `messages`. Each message has a `role` and `content`.

- `system`: instructions about how the model should behave
- `user`: the task or question
- `assistant`: previous model responses, useful in longer conversations

In [11]:
example_payload = {
    "model": "google/gemini-2.5-flash-lite",
    "messages": [
        {
            "role": "system",
            "content": "You are a concise assistant for digital humanities students.",
        },
        {
            "role": "user",
            "content": "Explain why JSON is useful for archive metadata in two sentences.",
        },
    ],
    "temperature": 0.2,
    "max_tokens": 200,
}

print(json.dumps(example_payload, indent=2, ensure_ascii=False))

{
  "model": "google/gemini-2.5-flash-lite",
  "messages": [
    {
      "role": "system",
      "content": "You are a concise assistant for digital humanities students."
    },
    {
      "role": "user",
      "content": "Explain why JSON is useful for archive metadata in two sentences."
    }
  ],
  "temperature": 0.2,
  "max_tokens": 200
}


## Optional: Check Available OpenRouter Models

Model availability and pricing change over time. This optional cell asks OpenRouter for model metadata and prints a few models.

If this cell fails, continue with the default model below and change it only if your API request reports that the model is unavailable.

In [13]:
def openrouter_headers(api_key=OPENROUTER_API_KEY):
    headers = {
        "Content-Type": "application/json",
        "HTTP-Referer": "https://github.com/ValRCS/RTU_Python_DE0800", # not required but nice to do
        "X-OpenRouter-Title": "RTU Python DE0800 JSON API lesson",
    }
    if api_key:
        headers["Authorization"] = f"Bearer {api_key}"
    return headers


def list_openrouter_models(limit=8, supported_parameter=None):
    if not HAS_OPENROUTER_KEY:
        print("Skipping model listing because no API key is available.")
        return []

    params = {}
    if supported_parameter:
        params["supported_parameters"] = supported_parameter

    response = requests.get(
        "https://openrouter.ai/api/v1/models",
        headers=openrouter_headers(),
        params=params,
        timeout=30,
    )
    response.raise_for_status()
    data = response.json().get("data", [])

    for model in data[:limit]:
        model_id = model.get("id")
        name = model.get("name")
        context = model.get("context_length")
        parameters = ", ".join(model.get("supported_parameters") or [])
        print(f"{model_id} | {name} | context={context} | {parameters}")

    return data

# Uncomment if you want to inspect models before choosing one.
available_models = list_openrouter_models(supported_parameter="response_format")
available_models[:5] # so you can get info on various models - so metadata about model capabilites
# and surprise the data comes back as JSON! :)

minimax/minimax-m2.5:free | MiniMax: MiniMax M2.5 (free) | context=204800 | include_reasoning, max_tokens, reasoning, response_format, seed, stop, temperature, tools
google/gemma-4-26b-a4b-it:free | Google: Gemma 4 26B A4B  (free) | context=262144 | include_reasoning, max_tokens, reasoning, response_format, seed, temperature, tool_choice, tools, top_p
google/gemma-4-31b-it:free | Google: Gemma 4 31B (free) | context=262144 | include_reasoning, max_tokens, reasoning, response_format, seed, temperature, tool_choice, tools, top_p
mistralai/magistral-medium-2509 | Mistral: Magistral Medium 2509 | context=0 | frequency_penalty, include_reasoning, max_tokens, presence_penalty, reasoning, response_format, seed, stop, structured_outputs, temperature, tool_choice, tools, top_p
nvidia/nemotron-nano-9b-v2:free | NVIDIA: Nemotron Nano 9B V2 (free) | context=32000 | include_reasoning, max_tokens, reasoning, response_format, seed, structured_outputs, temperature, tool_choice, tools, top_p
cognitivec

[{'id': 'minimax/minimax-m2.5:free',
  'canonical_slug': 'minimax/minimax-m2.5-20260211',
  'hugging_face_id': 'MiniMaxAI/MiniMax-M2.5',
  'name': 'MiniMax: MiniMax M2.5 (free)',
  'created': 1770908502,
  'description': 'MiniMax-M2.5 is a SOTA large language model designed for real-world productivity. Trained in a diverse range of complex real-world digital working environments, M2.5 builds upon the coding expertise of M2.1...',
  'context_length': 204800,
  'architecture': {'modality': 'text->text',
   'input_modalities': ['text'],
   'output_modalities': ['text'],
   'tokenizer': 'Other',
   'instruct_type': None},
  'pricing': {'prompt': '0', 'completion': '0'},
  'top_provider': {'context_length': 196608,
   'max_completion_tokens': 8192,
   'is_moderated': True},
  'per_request_limits': None,
  'supported_parameters': ['include_reasoning',
   'max_tokens',
   'reasoning',
   'response_format',
   'seed',
   'stop',
   'temperature',
   'tools'],
  'default_parameters': {'temperat

## A Reusable OpenRouter Function

The next function sends a chat completion request and returns the full JSON response.

We keep the full response because it contains more than text: model name, usage information, and sometimes other metadata.

If you get a model error, change `DEFAULT_MODEL` after checking current OpenRouter models.

In [14]:
DEFAULT_MODEL = "google/gemini-2.5-flash-lite"


def openrouter_chat(
    system_prompt,
    user_prompt,
    model=DEFAULT_MODEL,
    temperature=0.2,
    max_tokens=700,
    response_format=None,
    api_key=OPENROUTER_API_KEY,
):
    """Send one chat request to OpenRouter and return the parsed JSON response."""
    if not api_key:
        print("Skipping API call because no OpenRouter API key is available.")
        return None

    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        "temperature": temperature,
        "max_tokens": max_tokens,
    }

    if response_format is not None:
        payload["response_format"] = response_format

    try:
        # here you actually make an API request over internet to service provider
        # in this case openrouter.ai
        response = requests.post(
            "https://openrouter.ai/api/v1/chat/completions",
            headers=openrouter_headers(api_key),
            json=payload,
            timeout=60, # we wait 60 seconds max for response
        )
        response_json = response.json()
    except requests.exceptions.RequestException as error:
        print("Network or request error:", error)
        return None
    except json.JSONDecodeError:
        print("The server did not return valid JSON.")
        print(response.text[:1000])
        return None

    if not response.ok:
        print("API error status:", response.status_code)
        print(json.dumps(response_json, indent=2, ensure_ascii=False)[:1500])
        return None

    return response_json


def get_message_text(response_json):
    """Extract assistant message text from a chat completion response."""
    if not response_json:
        return None
    # note theoretically you could get IndexError here
    # why because [0] the first message might not exists - rare but sometimes response is empty
    return response_json["choices"][0]["message"]["content"]

## First LLM API Call

This is the smallest useful API example in this notebook. It asks the model to explain JSON for humanities research.

If you do not have an API key, this cell prints a skip message.

In [15]:
system_prompt = "You explain technical ideas clearly to beginning digital humanities students."
user_prompt = "In two short sentences, explain why JSON is useful for humanities research data."

first_response = openrouter_chat(system_prompt, user_prompt)
first_text = get_message_text(first_response)

if first_text:
    print(first_text)

JSON is useful for humanities research data because its structured format makes it easy to organize and access complex information like historical texts, metadata, or interview transcripts. This organization allows researchers to efficiently query and analyze their data, uncovering patterns and insights that might otherwise be hidden.


## Inspect the API Response JSON

The visible answer is inside a larger JSON object. This is why learning to navigate dictionaries and lists matters.

In [16]:
if first_response:
    compact_view = {
        "id": first_response.get("id"),
        "model": first_response.get("model"),
        "message": first_response["choices"][0]["message"],
        "usage": first_response.get("usage"),
    }
    print(json.dumps(compact_view, indent=2, ensure_ascii=False))
else:
    print("No API response to inspect yet.")

{
  "id": "gen-1779286831-JWmRj4wTtKi9mScu1BWn",
  "model": "google/gemini-2.5-flash-lite",
  "message": {
    "role": "assistant",
    "content": "JSON is useful for humanities research data because its structured format makes it easy to organize and access complex information like historical texts, metadata, or interview transcripts. This organization allows researchers to efficiently query and analyze their data, uncovering patterns and insights that might otherwise be hidden.",
    "refusal": null,
    "reasoning": null
  },
  "usage": {
    "prompt_tokens": 26,
    "completion_tokens": 52,
    "total_tokens": 78,
    "cost": 2.34e-05,
    "is_byok": false,
    "prompt_tokens_details": {
      "cached_tokens": 0,
      "cache_write_tokens": 0,
      "audio_tokens": 0,
      "video_tokens": 0
    },
    "cost_details": {
      "upstream_inference_cost": 2.34e-05,
      "upstream_inference_prompt_cost": 2.6e-06,
      "upstream_inference_completions_cost": 2.08e-05
    },
    "comple

In [20]:
# the beauty of this approach is that we can trivially try another provider OR model
# let's try the latest Gemini Flash 3.5 which came out on 19th of May 2026
GEMINI_FLASH35 = "google/gemini-3.5-flash" # just a string from https://openrouter.ai/google/gemini-3.5-flash/api
second_response = openrouter_chat(system_prompt,
                                  user_prompt,
                                  model=GEMINI_FLASH35,
                                  max_tokens=5000)
second_text = get_message_text(second_response)

if second_text:
    print(second_text)

JSON is highly useful because its simple, text-based format is easy for both human researchers to read and computers to analyze. Its flexible structure also perfectly accommodates the complex, nested relationships often found in humanities data, like historical networks or literary archives.


In [21]:
# let's save both texts to text files
with open("first_response.txt", "w", encoding="utf-8") as file:
    file.write(first_text)
with open("second_response.txt", "w", encoding="utf-8") as file:
    file.write(second_text)

## Humanities Text Samples

For Colab, the notebook includes short embedded texts. Locally, it can also use `data/veidenbaums_clean.txt` from this repository if it is available.

In [22]:
embedded_samples = [
    {
        "id": "archive_note_001",
        "title": "Student letter sample",
        "text": (
            "Anna writes from Riga to her sister Marta in Jelgava. "
            "She describes lectures, the railway journey, and her worry that books are becoming too expensive."
        ),
    },
    {
        "id": "newspaper_note_001",
        "title": "Newspaper notice sample",
        "text": (
            "The city library announces a public lecture about folk songs, printing history, "
            "and new methods for cataloging local newspapers."
        ),
    },
    {
        "id": "memoir_note_001",
        "title": "Memoir sample",
        "text": (
            "The narrator remembers arriving in Riga during winter, carrying a small suitcase, "
            "and searching for a room near the university."
        ),
    },
]


def load_local_veidenbaums_excerpt(max_characters=1200):
    candidates = [
        Path("data/veidenbaums_clean.txt"),
        Path("../data/veidenbaums_clean.txt"),
    ]
    for path in candidates:
        if path.exists():
            text = path.read_text(encoding="utf-8")[:max_characters]
            return {
                "id": "veidenbaums_local_excerpt",
                "title": "Local Veidenbaums excerpt",
                "text": text,
                "source_path": str(path),
            }
    return None

local_sample = load_local_veidenbaums_excerpt()
if local_sample:
    print("Loaded local sample from", local_sample["source_path"])
    selected_sample = local_sample
else:
    print("Using embedded sample because local data file was not found.")
    selected_sample = embedded_samples[0]

print("Sample title:", selected_sample["title"])
print(selected_sample["text"][:500])

Using embedded sample because local data file was not found.
Sample title: Student letter sample
Anna writes from Riga to her sister Marta in Jelgava. She describes lectures, the railway journey, and her worry that books are becoming too expensive.


## Ask for Structured JSON

Natural-language answers are easy to read but harder to analyze with code.

For research workflows, we often want the model to return structured JSON. OpenRouter supports `response_format={"type": "json_object"}` for models that support JSON mode. We should still explicitly tell the model to return JSON only.

In [23]:
analysis_system_prompt = """
You are a careful digital humanities research assistant.
Analyze the supplied text and return only valid JSON.
Do not wrap the JSON in Markdown.
Use this exact JSON structure:
{
  "summary": "one sentence summary",
  "people": ["person names mentioned in the text"],
  "places": ["place names mentioned in the text"],
  "themes": ["short theme labels"],
  "uncertainty": "short note about possible ambiguity or missing context"
}
""".strip()

analysis_user_prompt = f"Title: {selected_sample['title']}\n\nText:\n{selected_sample['text']}"

structured_response = openrouter_chat(
    analysis_system_prompt,
    analysis_user_prompt,
    temperature=0.1,
    max_tokens=700,
    response_format={"type": "json_object"},
)

structured_text = get_message_text(structured_response)
if structured_text:
    print(structured_text)

{
  "summary": "Anna writes to her sister Marta from Riga, discussing lectures, travel, and concerns about book prices.",
  "people": ["Anna", "Marta"],
  "places": ["Riga", "Jelgava"],
  "themes": ["education", "travel", "economics"],
  "uncertainty": "The specific year or time period of the letter is not mentioned, which could affect the context of book prices and lectures."
}


## Parse the LLM JSON Output

The API response is JSON. Inside it, the assistant message may also be JSON text. That means we parse twice:

1. `response.json()` parses the API response
2. `json.loads(message_text)` parses the assistant's JSON answer

In [24]:
def parse_llm_json(text):
    """Parse JSON returned by an LLM, with a small cleanup for Markdown fences."""
    if not text:
        return None

    cleaned = text.strip()
    if cleaned.startswith("```json"):
        cleaned = cleaned[len("```json"):].strip()
    elif cleaned.startswith("```"):
        cleaned = cleaned[len("```"):].strip()
    if cleaned.endswith("```"):
        cleaned = cleaned[:-3].strip()

    return json.loads(cleaned)

try:
    parsed_analysis = parse_llm_json(structured_text)
    print(type(parsed_analysis))
    print(json.dumps(parsed_analysis, indent=2, ensure_ascii=False))
except json.JSONDecodeError as error:
    parsed_analysis = None
    print("Could not parse the model output as JSON:", error)

<class 'dict'>
{
  "summary": "Anna writes to her sister Marta from Riga, discussing lectures, travel, and concerns about book prices.",
  "people": [
    "Anna",
    "Marta"
  ],
  "places": [
    "Riga",
    "Jelgava"
  ],
  "themes": [
    "education",
    "travel",
    "economics"
  ],
  "uncertainty": "The specific year or time period of the letter is not mentioned, which could affect the context of book prices and lectures."
}


## Navigate the Parsed Result

Once the result is parsed, it is normal Python data. You can access fields by key.

In [25]:
if parsed_analysis:
    print("Summary:", parsed_analysis["summary"])
    print("Places:", parsed_analysis["places"])
    print("Themes:", parsed_analysis["themes"])
else:
    print("No parsed analysis available yet.")

Summary: Anna writes to her sister Marta from Riga, discussing lectures, travel, and concerns about book prices.
Places: ['Riga', 'Jelgava']
Themes: ['education', 'travel', 'economics']


## Exercise: Modify the Prompt

Change one thing and rerun the structured JSON cells:

- ask for `dates`
- ask for `emotions`
- ask for `research_questions`
- ask the model to quote evidence from the text
- switch to another embedded sample

When you add a new JSON field in the prompt, remember to access it by its new key in Python.

In [26]:
# Try changing this number to 0, 1, or 2.
sample_number = 1
selected_sample = embedded_samples[sample_number]

print("Selected:", selected_sample["title"])
print(selected_sample["text"])

Selected: Newspaper notice sample
The city library announces a public lecture about folk songs, printing history, and new methods for cataloging local newspapers.


## Mini Batch: Analyze Several Short Texts

A common research workflow is to repeat the same prompt across multiple documents.

This example processes only three small samples to keep cost and time low.

In [27]:
def analyze_sample_to_json(sample, model=DEFAULT_MODEL):
    user_prompt = f"Document ID: {sample['id']}\nTitle: {sample['title']}\n\nText:\n{sample['text']}"
    response = openrouter_chat(
        analysis_system_prompt,
        user_prompt,
        model=model,
        temperature=0.1,
        max_tokens=700,
        response_format={"type": "json_object"},
    )
    message_text = get_message_text(response)
    if not message_text:
        return None

    result = parse_llm_json(message_text)
    result["document_id"] = sample["id"]
    result["title"] = sample["title"]
    return result

RUN_MINI_BATCH = HAS_OPENROUTER_KEY

batch_results = []
if RUN_MINI_BATCH:
    for sample in embedded_samples:
        print("Analyzing", sample["id"])
        try:
            result = analyze_sample_to_json(sample)
            if result:
                batch_results.append(result)
        except Exception as error:
            print("Failed on", sample["id"], "because", error)
else:
    print("Mini batch skipped because no API key is available.")

print("Results collected:", len(batch_results))

Analyzing archive_note_001
Analyzing newspaper_note_001
Analyzing memoir_note_001
Results collected: 3


In [28]:
if batch_results:
    print(json.dumps(batch_results, indent=2, ensure_ascii=False))

    if pd is not None:
        display(pd.DataFrame(batch_results))
else:
    print("No batch results to display yet.")

[
  {
    "summary": "Anna writes from Riga to her sister Marta in Jelgava about lectures, travel, and the rising cost of books.",
    "people": [
      "Anna",
      "Marta"
    ],
    "places": [
      "Riga",
      "Jelgava"
    ],
    "themes": [
      "education",
      "travel",
      "economics"
    ],
    "uncertainty": "The specific year or time period of the letter is not mentioned.",
    "document_id": "archive_note_001",
    "title": "Student letter sample"
  },
  {
    "summary": "The city library is hosting a public lecture covering folk songs, printing history, and new methods for cataloging local newspapers.",
    "people": [],
    "places": [
      "city library"
    ],
    "themes": [
      "folk songs",
      "printing history",
      "newspaper cataloging"
    ],
    "uncertainty": "No specific individuals or locations beyond the general 'city library' are mentioned.",
    "document_id": "newspaper_note_001",
    "title": "Newspaper notice sample"
  },
  {
    "summ

,summary,people,places,themes,uncertainty,document_id,title
0,Anna writes from Riga to her sister Marta in J...,"[Anna, Marta]","[Riga, Jelgava]","[education, travel, economics]",The specific year or time period of the letter...,archive_note_001,Student letter sample
1,The city library is hosting a public lecture c...,[],[city library],"[folk songs, printing history, newspaper catal...",No specific individuals or locations beyond th...,newspaper_note_001,Newspaper notice sample
2,The narrator recalls arriving in Riga during w...,[narrator],[Riga],"[arrival, winter, housing search, university]",The specific university is not mentioned.,memoir_note_001,Memoir sample


## Save Batch Results

Structured output becomes more useful once saved. The result file can be opened later in Python, a text editor, or another data tool.

In [29]:
if batch_results:
    output_path = Path("llm_humanities_results.json")
    with output_path.open("w", encoding="utf-8") as file:
        json.dump(batch_results, file, indent=2, ensure_ascii=False)
    print("Saved results to", output_path.resolve())
else:
    print("No batch results to save yet.")

Saved results to /content/llm_humanities_results.json


In [30]:
# if we have pandas dataframe we can save it as CSV or Excel or something else
# let's save as CSV
if batch_results and pd is not None:
    df = pd.DataFrame(batch_results)
    df.to_csv("llm_humanities_results.csv", index=False)

## Discussion: Responsible Use

LLMs can help with exploration, annotation, translation, and metadata generation, but they are not neutral research instruments.

For humanities work, pay attention to:

- uncertainty: the model may infer more than the text supports
- bias: model training data may overrepresent some languages, periods, or perspectives
- privacy: do not send sensitive or copyrighted material unless you have permission
- reproducibility: save prompts, model names, parameters, and dates
- cost: test on small samples before processing larger corpora

## Bonus: Compare Models

Different models may give different answers. Use a very small text first, because model comparisons multiply cost.

Before changing the list, check OpenRouter's current model page or run the model listing cell above.

In [32]:
BONUS_RUN_MODEL_COMPARISON = True
models_to_compare = [
    DEFAULT_MODEL,
    # Add another model ID here after checking availability, for example:
    # "google/gemini-2.5-flash",
    "google/gemini-3.5-flash",
    "anthropic/claude-opus-4.7-fast"
]

if BONUS_RUN_MODEL_COMPARISON and HAS_OPENROUTER_KEY:
    comparison_results = []
    for model in models_to_compare:
        print("Trying model:", model)
        response = openrouter_chat(
            analysis_system_prompt,  # and this string would be your instructions for analysis
            analysis_user_prompt, # so this part would be your text to analyse
            model=model,
            temperature=0.1, # adjust this for more variety, less temperature more fixed answers
            max_tokens=700, # adjust to higher for loner response , some models ignore this
            response_format={"type": "json_object"},
        )
        text = get_message_text(response)
        comparison_results.append({"model": model, "response": text})

    for item in comparison_results:
        print("\nMODEL:", item["model"])
        print(item["response"])
else:
    print("Model comparison is off. Set BONUS_RUN_MODEL_COMPARISON = True to run it.")

Trying model: google/gemini-2.5-flash-lite
Trying model: google/gemini-3.5-flash
Trying model: anthropic/claude-opus-4.7-fast

MODEL: google/gemini-2.5-flash-lite
{
  "summary": "Anna writes to her sister Marta about lectures, travel, and the rising cost of books.",
  "people": ["Anna", "Marta"],
  "places": ["Riga", "Jelgava"],
  "themes": ["education", "travel", "economics"],
  "uncertainty": "The specific year or time period of the letter is not mentioned."
}

MODEL: google/gemini-3.5-flash
{
  "summary": "Anna writes a letter from Riga to her sister Marta in Jelgava, sharing details about her lectures, a railway journey, and her concerns over the rising cost of books.",
  "people": [
    "Anna",
    "Marta"
  ],
  "places": [
    "Riga",
    "Jelgava"
  ],
  "themes": [
    "Student life",
    "Education",
    "Travel",
    "Economic concerns",
    "Correspondence"
  ],
  "uncertainty": "The historical period, specific date of the letter, and Anna's field of study are not specified

## Bonus: Repair Invalid JSON

Even with careful prompting, some models may return Markdown fences or malformed JSON. The first repair step should be deterministic cleanup, not another API call.

The `parse_llm_json` function above already removes simple Markdown fences. If JSON is still invalid, inspect the text before deciding what to do.

In [ ]:
example_fenced_json = """
```json
{
  "summary": "A sample answer wrapped in Markdown.",
  "people": [],
  "places": ["Riga"],
  "themes": ["metadata"],
  "uncertainty": "This is only an example."
}
```
"""

print(parse_llm_json(example_fenced_json))

## Bonus: Process a Folder of Text Files

This is a simplified version of a larger corpus workflow. It is useful for local Jupyter use after you understand the smaller examples.

Keep `max_files` small while developing your prompt.

In [ ]:
def analyze_text_folder(
    folder_path,
    pattern="*.txt",
    max_files=3,
    output_file="folder_analysis_results.json",
    model=DEFAULT_MODEL,
):
    folder = Path(folder_path)
    if not folder.exists():
        print("Folder not found:", folder)
        return []

    files = sorted(folder.rglob(pattern))[:max_files]
    print(f"Found {len(files)} files to process.")

    results = []
    for path in files:
        text = path.read_text(encoding="utf-8", errors="replace")[:2500]
        sample = {"id": path.stem, "title": path.name, "text": text}
        print("Analyzing", path.name)
        result = analyze_sample_to_json(sample, model=model)
        if result:
            result["source_path"] = str(path)
            results.append(result)

    if results:
        with Path(output_file).open("w", encoding="utf-8") as file:
            json.dump(results, file, indent=2, ensure_ascii=False)
        print("Saved", output_file)

    return results

BONUS_RUN_FOLDER_ANALYSIS = False
if BONUS_RUN_FOLDER_ANALYSIS and HAS_OPENROUTER_KEY:
    folder_results = analyze_text_folder("data", max_files=2)
else:
    print("Folder analysis is off. Set BONUS_RUN_FOLDER_ANALYSIS = True to run it.")

## Bonus: Translation and Noisy Historical Text

LLMs can be useful for exploratory translation and OCR cleanup, especially when working with historical newspapers, letters, or printed material.

Treat the result as a research aid, not as a final scholarly translation. Always keep the original text.

In [ ]:
historical_sample = """
Riga, den 15. Marz 1918. Die Zeitung berichtet uber Schulen, Preise und den Eisenbahnverkehr.
Einige Buchstaben sind unsicher, weil die OCR-Erkennung aus einer alten Druckschrift stammt.
""".strip()

translation_prompt = """
You are helping with exploratory analysis of historical German text.
Translate the text into modern English.
If the text appears to contain OCR errors, silently correct only obvious errors.
Return JSON with keys: translation, possible_ocr_errors, uncertainty.
Return only valid JSON.
""".strip()

BONUS_RUN_TRANSLATION = False
if BONUS_RUN_TRANSLATION and HAS_OPENROUTER_KEY:
    translation_response = openrouter_chat(
        translation_prompt,
        historical_sample,
        temperature=0.1,
        max_tokens=700,
        response_format={"type": "json_object"},
    )
    translation_text = get_message_text(translation_response)
    print(translation_text)
else:
    print("Translation bonus is off. Set BONUS_RUN_TRANSLATION = True to run it.")

## Bonus: Image Input to Multimodal Models

Some OpenRouter models can accept images. This is useful for page scans, manuscripts, posters, or printed historical material.

Not all models support images. Check the model metadata first and keep image experiments small.

In [ ]:
# This is a template only. It is not run by default.

import base64
import mimetypes


def encode_image_as_data_url(image_path):
    path = Path(image_path)
    mime_type, _ = mimetypes.guess_type(path)
    if not mime_type or not mime_type.startswith("image/"):
        raise ValueError("Unsupported image type")

    image_base64 = base64.b64encode(path.read_bytes()).decode("utf-8")
    return f"data:{mime_type};base64,{image_base64}"


def openrouter_image_chat(image_path, prompt, model=DEFAULT_MODEL, api_key=OPENROUTER_API_KEY):
    if not api_key:
        print("Skipping image API call because no OpenRouter API key is available.")
        return None

    data_url = encode_image_as_data_url(image_path)
    payload = {
        "model": model,
        "messages": [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {"type": "image_url", "image_url": {"url": data_url}},
                ],
            }
        ],
        "max_tokens": 700,
    }

    response = requests.post(
        "https://openrouter.ai/api/v1/chat/completions",
        headers=openrouter_headers(api_key),
        json=payload,
        timeout=90,
    )
    response.raise_for_status()
    return response.json()

print("Image helper functions defined. Choose an image-capable model before using them.")

## Recap

You have practiced the main workflow:

1. represent research data in Python dictionaries and lists
2. convert Python data to and from JSON
3. build an API request as JSON
4. send a request to OpenRouter
5. inspect the response JSON
6. ask the model for structured JSON
7. parse and save results

For a real research project, start small: one text, one prompt, one model, and one output format. Expand only after the small version works.